### **Student Information**
GitHub ID: ChinnakritTete

Kaggle name: Chinnakrit Kritsaneepaiboon

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development

**Task:** read each post and pick one of 6 emotions: anger, disgust, fear, joy, sadness, surprise.

**Score used by Kaggle:** macro F1. It scores each emotion on its own, then takes the plain average. So a small emotion (like fear) counts as much as a big one (like joy).

**Who did what.** I used AI (Claude Code) for most of the coding. The AI wrote the code, ran the laptop parts, and tested the settings described below. I ran the GPU training sessions on Kaggle, uploaded every submission, reported errors back to the AI, and decided which suggestions to keep or reject (all listed in 1.4).

**Final model:** a blend that averages the answers of four models:
1. **RoBERTa-large**, fine-tuned (a big pre-trained language model, trained further on our posts),
2. **ELECTRA-base**, fine-tuned 3 times with different random starts (seeds),
3. **DeBERTa-v3-large**, fine-tuned,
4. a **Linear SVM** trained on word counts (TF-IDF) and sentence embeddings.

| | Score |
|---|---|
| Honest score (5-fold out-of-fold macro F1 on all 3,360 training posts, explained in 1.3) | **0.619** (about **0.614** after a small correction, see 1.3) |
| Kaggle public score of the final file | 0.5953 |
| My best public score of all submissions | 0.6089 |
| **Kaggle private leaderboard (final, screenshot at the top of this notebook)** | **0.6347, rank 53** (up 20 places from the public board) |

---

### 1.1 Preprocessing Steps

1. **Load and join the four files.** The text (`final_posts.json`) and the labels (`emotion.csv`) are in different files. A model can only learn when each text sits next to its label, so the code joins them by `post_id`. It starts from `data_identification.csv`, so every post appears once and always in the same order.
   * Result: **4,800 posts = 3,360 training + 1,440 test**.
   * The real file names differ from the task description (`final_posts.json`, `sampleSubmission.csv`), so the AI checked them before writing the code.
2. **Data checks.** No text is missing and no `post_id` appears twice. Every training post has a label, and no test post has one.
   * 50 texts appear twice in the training set, and 16 of them have **different** labels (the same words were labelled as two emotions). They were kept, because each one is a real label.
   * 59 test texts also appear in the training set. For 95% of them, the model already predicts the training label (checked in 1.3).
3. **What the data showed, and how it guided later choices.**
   * Share of each emotion: joy 26.0%, anger 20.1%, sadness 18.7%, disgust 13.5%, surprise 13.1%, **fear only 8.6%**. Macro F1 counts small emotions as much as big ones, so the small ones get more weight during training (1.3).
   * Length: the middle post (median) has 13 words, the longest has 508 words. 99% of posts are 103 tokens (word pieces) or fewer, so a **limit of 128 tokens** keeps almost every post whole.
4. **Cleaning is different for each kind of model:**
   * **For TF-IDF:** lowercase, replace links with the word `url`, remove extra spaces. Common words, punctuation and emoji are deliberately **kept**, and words are **not** cut down to their root, because emotion often lives in exactly these parts (`not`, `!!!`, `:(`). Masks like `[NAME]` and `[RELIGION]` are kept as words.
   * **For the transformers and sentence encoders:** the **raw text**. These models learned from normal text with capitals and punctuation, so cleaning would remove information they can use.
5. **I checked that the submission format works on Kaggle.** The AI wrote the code that saves predictions in the row order of `sampleSubmission.csv`. My first upload failed with "ID column id not found", because the sample file calls the column `post_id`. I reported the error, the AI renamed the column to **`id`**, and I uploaded the fixed file, which Kaggle accepted.

### 1.2 Feature Engineering Steps

Features are the numbers a model reads instead of the raw text.

**A. Features for the SVM (made by hand):**

1. **TF-IDF on words:** single words and word pairs, kept only if they appear in at least 2 posts (`min_df=2`). TF-IDF gives a word a high number when it is common in this post but rare in other posts. It uses the log of the count (`sublinear_tf=True`), so one word repeated many times cannot take over a post.
2. **TF-IDF on pieces of words:** 2–5 letters long (`char_wb`). Reddit spelling changes a lot (`sooo`, `lmaooo`, `f*ck`), but the pieces still match the normal word.
3. **Sentence embeddings** from two pre-trained models, used as they are: `all-MiniLM-L6-v2` (384 numbers) and `all-mpnet-base-v2` (768 numbers). An embedding is a list of numbers that describes the *meaning* of a sentence, so it works even when the words are different ("I can't believe it" ≈ surprise). They are saved to disk, because making them takes minutes on a CPU.
4. **Combining them:** the two TF-IDF blocks (each row scaled to length 1) sit side by side with the embeddings × 0.5. The AI tested 0.5, 0.7 and 1.0 in cross-validation, and 0.5 scored best. Total: **36,184 columns**.
5. **No leakage** (leakage = the model secretly seeing the answers): inside every cross-validation fold, the TF-IDF word list is rebuilt from the training part only, so the held-out part never shapes it.

| SVM features (5-fold CV, macro F1) | Score |
|---|---|
| Word TF-IDF only | 0.463 |
| Word + letter-piece TF-IDF | 0.510 |
| + MiniLM + mpnet embeddings | **0.569** |

**B. Features for the fine-tuned transformers (learned by the model):** each model cuts the text into its own word pieces (max 128) and learns its own features during training. The work here was making the input correct and fast:
* Posts of similar length go into the same batch, so less padding (empty filler) is wasted.
* A **tokenizer check** stops a run early if the text is cut wrongly. The text must decode back to the original, and fewer than 1% of pieces may be "unknown". The AI added it after DeBERTa's first run, where the tokenizer was silently built wrong (see 1.3).

### 1.3 Explanation of Your Model

**How the honest score was calculated (5-fold cross-validation).** I used AI (Claude Code) to calculate it. The AI wrote the code: it ran the SVM and blend parts on my laptop, and I ran the transformer part on Kaggle's GPU. My part was to ask for an honest check, to understand the principle below, and to decide to trust this score over the public leaderboard.

**The principle, step by step:**
1. **A model must be tested on posts it never saw.** If I score a model on its own training posts, it can just remember them, so the score looks much better than it really is.
2. **So the AI split the 3,360 training posts into 5 parts.** It trained on 4 parts and predicted the 5th, and repeated this 5 times, so each part is held out once.
3. **Every post then gets exactly one prediction from a model that never saw it.** This is called an **out-of-fold (OOF)** prediction.
4. **The macro F1 over all 3,360 OOF predictions is the honest score.** It uses every training post as a test, so it is much steadier than a score on 504 posts or on the part of the test set that the public leaderboard uses.
5. **Any choice made by looking at this score makes it a little too good.** When I choose settings (like blend weights) because they score best on these posts, part of the gain is luck on these posts. So the AI also measured this extra: choose on 3 folds, score on the other 2 (see "The blend" below).

**Model 1: Linear SVM** (`LinearSVC`, C = 0.3, `class_weight="balanced"`). It learns one weight for each feature and each emotion. It was chosen because it was the best classifier in the Master notebook.
* `class_weight="balanced"` gives rare emotions more weight, which raises macro F1.
* C = 0.3 holds the model back more than the default (1.0). The AI tested 0.3 / 0.5 / 1.0, and 0.3 scored best. With 36,184 columns and only 3,360 posts, the model needs this, or it memorizes the training posts.

**Model 2: fine-tuned transformers.** A transformer is a big language model that was first trained on a lot of general text. Fine-tuning means training **all** of its weights a bit more on our labelled posts, so it learns which phrases show each emotion. I decided to use only **general** pre-trained models and to avoid models already trained on emotion labels, because such a model may have seen these exact posts.
* The AI prepared the training notebook, and I ran it on a **Kaggle T4 GPU** with **5-fold cross-validation**. The test prediction is the average of the 5 models.
* Settings: AdamW (weight decay 0.01), learning rate that rises over the first 10% of steps and then falls slowly to 0, gradient clipping 1.0, batch 16, 4 epochs, mixed precision (fp16, faster) with 32-bit master weights.

| Fine-tuned model | Learning rate | OOF macro F1 (3,360 posts) |
|---|---|---|
| RoBERTa-large | 1e-5 | 0.602 |
| ELECTRA-base, 3 seeds averaged | 5e-5 | 0.600 |
| DeBERTa-v3-large | 1.5e-5 | 0.558 |

**The blend.** The SVM gives scores, not probabilities, so they are turned into probability-like numbers with `softmax(2 × score)`. Then all four models are averaged with weights. The AI tested 80 weight combinations and kept the one with the best OOF macro F1 on all 3,360 posts:

| RoBERTa-large | ELECTRA-base ×3 | DeBERTa-v3-large | SVM | OOF macro F1 |
|---|---|---|---|---|
| 0.29 | 0.29 | 0.14 | 0.29 | **0.619** |

* **Why blending helps:** each model alone gets about 0.60, but they make some different mistakes, so averaging cancels part of them.
* **How honest is 0.619?** Picking the best of 80 combinations makes the score look a bit too good. The AI tested picking the weights on 3 folds, and scoring them on the other 2. The score was too good by about +0.005, so the honest score is **about 0.614**.

**Results per emotion (OOF):** joy is the easiest (F1 0.769). **Disgust is the hardest (0.438)**, and the model mostly mixes it up with anger. Both react to something bad and use the same words. The only difference is whether the writer feels *sick* (disgust) or *attacks* (anger), and short posts often don't show that.

**How I got here, and what I learned:**

| Step | Honest score | Public score |
|---|---|---|
| 1. TF-IDF + embeddings + SVM | 0.569 (5-fold CV) | 0.5649 |
| 2. + RoBERTa-base and ELECTRA-base, trained on my laptop CPU (85% of the data), with blend weights chosen on the other 504 posts | 0.648 on 504 posts, **too optimistic** | 0.6089 |
| 3. + RoBERTa-large (Kaggle GPU), and the laptop models trained again on all posts | about 0.61 | 0.6001 |
| 4. **Final:** all GPU models with 5-fold OOF, weights chosen on all 3,360 posts | **0.619 (about 0.614)** | 0.5953 |

1. **Testing on only 504 posts misled me (step 2).** The best epoch *and* the blend weight were both chosen on the same 504 posts. When I asked the AI to diagnose the 0.6089, it chose the weights on one half and scored them on the other half, and found that this added about **+0.036** of false confidence. That explains most of the drop from 0.648 to 0.6089. So I switched the final model to 5-fold OOF on all 3,360 posts.
2. **The public leaderboard is too noisy to rank my last three submissions.** The public score uses only part of the test posts, so it moves by chance by about **0.017–0.023** (one standard deviation, measured by the AI). My public scores (0.6089, 0.6001, 0.5953) differ by less than that, and the files differ on only about 170 of 1,440 posts. The AI also trained a classifier to tell training posts from test posts. It scored AUC **0.507** (0.5 = it cannot tell them apart), so the test posts are like the training posts. For these reasons, I trust the OOF score more than the public score.
3. **Two large models did not train enough, and one had a bug.**
   * DeBERTa's first run scored only 0.460. Without the `sentencepiece` package, `transformers` could not read DeBERTa's word list and silently used a different reader, so the words were cut wrongly. The AI found this, added the `sentencepiece` install and the tokenizer check, and I ran it again: it reached 0.558, but it was still improving at epoch 4.
   * ELECTRA-large (0.474) also learned too slowly. The learning rate (1–1.5e-5) is right for RoBERTa but too low for these models. I switched to ELECTRA-base with 5e-5, which trained well, and I left ELECTRA-large out of the blend.
4. **Ideas that were tested and rejected**, because they did not improve the honest OOF score:

| Idea | Honest result |
|---|---|
| Force the share of each predicted emotion to match the training shares | −0.001 |
| A separate decision threshold for each emotion | −0.0003 |
| Stacking (a logistic regression that learns how to combine the models) | 0.610 vs 0.619 |
| Copy the training label for the 59 test posts that also appear in training | already 95% the same, nothing to gain |

**I selected two final submissions for the private leaderboard** (Kaggle grades the better of the two):

| Pick | Kaggle submission | Public score | Honest score | Why I chose it |
|---|---|---|---|---|
| 1 | Submission_5, `submission_final_v2.csv` (final blend) | 0.5953 | **0.619 (about 0.614)** | Best honest score. It is tested on all 3,360 training posts, while the public score uses only part of the 1,440 test posts. |
| 2 | Submission_2, `submission_ft_robertaft_roberta_s7ft_electra.csv` (step 2) | 0.6089 | about 0.61 | Backup. It gives the same answer as the final blend on only **88.5%** of test posts. If the final blend is unlucky on the private test posts, this different file has its own chance. |

* I made both picks myself instead of leaving them to Kaggle. With no picks, Kaggle uses the highest public scores, and point 2 above shows those differ only by chance.
* **Final result:** my private score is **0.6347**, rank **53**, which is 20 places higher than on the public board. This is about 0.02 above my honest estimate (0.614) and about 0.04 above my public score (0.5953). Both gaps are about the size of the noise I measured (0.017–0.023), so I read this as the public board being a poor guide, not as proof that one file is better. The screenshot does not show which of my two picks Kaggle graded, so I do not claim that.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

**AI tool:** Claude Code (Anthropic's Claude) in VS Code. It wrote and ran most of the code on my laptop, prepared the notebooks I ran on Kaggle's GPU, and explained each result. I did the Kaggle uploads and GPU sessions myself.

The table shows the prompts where I hit a problem and made a decision to solve it.

| # | Problem | My prompt (shortened) | What the AI suggested | What I decided, and why |
|---|---|---|---|---|
| 1 | Kaggle rejected my first upload. | "Submission fail: ID column id not found." | Kaggle wants the column name `id`, not `post_id` like in the sample file. | **Accepted** the one-line fix and uploaded again. Kaggle accepted the file. |
| 2 | My baseline scored 0.5649, far below first place (0.6778). | "I got 0.5649, first place has 0.6778, aim to beat them." | Fine-tune transformers (RoBERTa-base, ELECTRA-base) on my CPU, keep 15% of posts aside for testing, and blend them with the SVM. Do **not** use public models already trained on emotion labels, because they may have seen these posts. | **Accepted** the plan and the fairness rule, because a model that has seen the test posts would give an unfair score. **Rejected** installing an extra package (`tiktoken`) into the course's locked environment just to try DeBERTa on CPU. |
| 3 | My own test said 0.648, but Kaggle gave only 0.6089. | "I got 0.6089, diagnose and improve it." | Choosing settings on the same 504 posts added +0.036 of false confidence, and the public score is noisy. Fixes: train again on all posts, average more models, use fixed weights. Matching the emotion shares was tested (−0.001). | **Accepted** training on all data and fixed weights, because they remove the false confidence. **Rejected** share matching, because it made the score worse. |
| 4 | My laptop CPU is too slow for large models. | "You can train on Kaggle, there is a GPU." / "Just tell me what to import and run." | A Kaggle notebook ready to import: 5-fold CV, large models, saves the OOF and test probabilities. It was first tested in a small CPU trial run. | **Proposed** Kaggle's free GPU myself, then **ran** the notebook there. It gave honest OOF scores on all 3,360 posts instead of 504. |
| 5 | The DeBERTa run crashed on Kaggle. | "[Kaggle log] Is it normal?" | DeBERTa's weights loaded as 16-bit numbers (fp16). The script moved on to RoBERTa-large, which trained fine (OOF 0.602). Fix: change the model to 32-bit after loading. | **Kept** RoBERTa-large in the blend, because it trained well, and **accepted** the fix for the next run. |
| 6 | The large models trained slowly and scored low (ELECTRA-large 0.474). | "[log] Is it normal?" (DeBERTa slow) / ELECTRA-large result | Both large models did not train enough (learning rate too low for them). Use ELECTRA-base with its known-good learning rate (5e-5) and 3 seeds on the GPU. | **Ran** ELECTRA-base ×3 (OOF 0.600) and kept it. **Dropped** ELECTRA-large (0.474), because it was much weaker. |
| 7 | My final model scored lower on the public board (0.5953). | "I got 0.5953 … final chance to diagnose and run on your own." | Tested the public score noise (SD 0.017–0.023), train/test difference (AUC 0.507), copying labels for repeated posts, per-emotion thresholds and stacking. None gave an honest gain. Stop and choose two final submissions. | **Stopped** adding models. The drop is within the public board's noise, and the extra CPU time could not give a gain big enough for the leaderboard to even show. |
| 8 | I had to choose which submissions Kaggle grades on the private board. | "Which one is my best honest score?" (screenshot of my Kaggle submissions) | The final blend has the best honest score. Submission_4 gives the same answer as it on 95.8% of posts, Submission_2 on 88.5%. Pick the final blend plus Submission_2 as a different backup. | **Selected** Submission_5 (best honest score) and Submission_2 (a different backup). I skipped Submission_4, because it is almost a copy of the final blend. |

**Prompts where I asked for explanations.** I asked these to make sure I understood my results, not only to get code. The last column is what I took away.

| # | My question (shortened) | What the AI explained | What I understood |
|---|---|---|---|
| A | "Why did 1st place get 0.6778 with the same limited data?" | It measured the label noise: 16 of 50 repeated texts disagree, so about 18% of labels are probably wrong, and even a perfect model would score only about **0.81**. It also simulated that the top team on a leaderboard of 30–100 equally good teams scores **+0.04–0.05** above its real level by luck alone. | The noisy labels are not what stopped me at 0.61, because there is still room up to about 0.81. First place is probably part luck and part a somewhat stronger model, not a completely different level. |
| B | "Use simple words to explain me the root cause, why others can beat that, what should I do to improve that." | All my models learn from the same 3,360 posts, so they make many of the same mistakes, and blending adds only about +0.02. Two large models were under-trained. Ways to improve: train DeBERTa-v3-large properly (higher learning rate, more epochs), try a model pre-trained on social-media text, or add an LLM as an extra voter. | A blend can only fix posts where the models disagree. To go higher, I need a model that is stronger or makes **different** mistakes, not more tuning of the same models. |

**Other prompts I used**: "Let's start the Kaggle part." (baseline plan: TF-IDF + SVM with 5-fold CV, accepted), "Prepare me the final submission." (blend weights chosen by OOF, accepted)

Overall, I kept an AI suggestion only when an honest check supported it: 5-fold OOF on all training posts, not the public leaderboard or a small test set. I rejected ideas that sounded good but did not improve that check (matching emotion shares, per-emotion thresholds, stacking, ELECTRA-large).

---


**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
### Add the code related to the preprocessing steps in cells inside this section
import json
import os
import re
import time

import numpy as np
import pandas as pd

SEED = 42
DATA_DIR = "data/kaggle/2026-dm-lab-2-private-competition"   # unzipped Kaggle data (git-ignored)

# 1. Load the four files. The text and the labels are in DIFFERENT files,
#    so nothing can be learned until they are joined on post_id.
ids = pd.read_csv(f"{DATA_DIR}/data_identification.csv")       # post_id, set (train / test)
labels = pd.read_csv(f"{DATA_DIR}/emotion.csv")                 # post_id, label (train rows only)
with open(f"{DATA_DIR}/final_posts.json", encoding="utf-8") as f:
    posts = pd.DataFrame(json.load(f))                          # post_id, text
sample_sub = pd.read_csv(f"{DATA_DIR}/sampleSubmission.csv")    # the format Kaggle expects

# 2. Merge on post_id. A left merge from `ids` keeps every post exactly once,
#    in the same order as data_identification.csv.
df = ids.merge(posts, on="post_id", how="left").merge(labels, on="post_id", how="left")

# 3. Validation: nothing missing, nothing duplicated, test rows have no labels.
print("posts per split:", df["set"].value_counts().to_dict())
print("missing text:", df["text"].isna().sum(), "| repeated post_id:", df["post_id"].duplicated().sum())
print("train rows without a label:", df.loc[df["set"] == "train", "label"].isna().sum(),
      "| test rows WITH a label:", df.loc[df["set"] == "test", "label"].notna().sum())

# 4. Light cleaning for the TF-IDF features only: lowercase, replace links with a
#    placeholder word, squeeze spaces. [NAME] / [RELIGION] masks are kept as words.
#    The sentence encoders get the raw text, because they were trained on normal casing.
def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"http\S+|www\.\S+", " url ", s)
    return re.sub(r"\s+", " ", s).strip()

df["clean"] = df["text"].map(clean_text)

# 5. Split back into the rows we learn from and the rows we must predict.
train = df[df["set"] == "train"].reset_index(drop=True)
test = df[df["set"] == "test"].reset_index(drop=True)
y = train["label"].to_numpy()
print("train:", train.shape, "| test:", test.shape)
print("label share in train:", train["label"].value_counts(normalize=True).round(3).to_dict())
print("words per post: median", train["text"].str.split().str.len().median(),
      "| max", train["text"].str.split().str.len().max())

posts per split: {'train': 3360, 'test': 1440}
missing text: 0 | repeated post_id: 0
train rows without a label: 0 | test rows WITH a label: 0
train: (3360, 5) | test: (1440, 5)
label share in train: {'joy': 0.26, 'anger': 0.201, 'sadness': 0.187, 'disgust': 0.135, 'surprise': 0.131, 'fear': 0.086}
words per post: median 13.0 | max 508


## 2. Feature Engineering Steps

In [ ]:
### Add the code related to the feature engineering steps in cells inside this section
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

# Feature set A: TF-IDF on words (1-2 grams) AND on letter pieces (2-5 characters).
# Character pieces help with Reddit spelling: "sooo", "lmaooo", "f*ck" still share
# pieces with the normal word. sublinear_tf uses log(count), so one repeated word
# cannot dominate a post. Fitted on training text only (no leakage).
def tfidf_blocks(fit_texts, other_texts):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    A = hstack([word.fit_transform(fit_texts), char.fit_transform(fit_texts)])
    B = hstack([word.transform(other_texts), char.transform(other_texts)])
    return normalize(A), normalize(B)          # each row scaled to length 1

# Feature set B: sentence embeddings from two pre-trained encoders (no training on
# our data, so no leakage). They capture meaning even when the words differ.
# Encoding takes a few minutes on CPU, so the vectors are cached in data/embeddings/
# (git-ignored). Delete those files if you change a model or the data.
ENCODERS = {"minilm": "sentence-transformers/all-MiniLM-L6-v2",
            "mpnet": "sentence-transformers/all-mpnet-base-v2"}

def cached_embeddings(key, model_name):
    path = f"data/embeddings/kaggle_{key}.npy"
    if os.path.exists(path):
        E = np.load(path)
        if E.shape[0] == len(df):              # same posts, same order
            return E
    t0 = time.time()
    E = SentenceTransformer(model_name).encode(df["text"].tolist(), batch_size=64,
                                               normalize_embeddings=True, show_progress_bar=False)
    os.makedirs("data/embeddings", exist_ok=True)
    np.save(path, E)
    print(f"encoded {key}: {E.shape} in {time.time() - t0:.0f}s")
    return E

E_all = np.hstack([cached_embeddings(k, m) for k, m in ENCODERS.items()])   # 384 + 768 = 1152 numbers per post
is_train = (df["set"] == "train").to_numpy()
E_train, E_test = E_all[is_train], E_all[~is_train]

# Combine A and B. EMB_WEIGHT scales the embedding columns relative to TF-IDF;
# 0.5 was the best of 0.5 / 0.7 / 1.0 in 5-fold cross-validation.
EMB_WEIGHT = 0.5

def build_features(fit_texts, other_texts, E_fit, E_other):
    A, B = tfidf_blocks(fit_texts, other_texts)
    X_fit = hstack([A, csr_matrix(E_fit * EMB_WEIGHT)]).tocsr()
    X_other = hstack([B, csr_matrix(E_other * EMB_WEIGHT)]).tocsr()
    return X_fit, X_other

X_train, X_test = build_features(train["clean"], test["clean"], E_train, E_test)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (3360, 36184) | X_test: (1440, 36184)


## 3. Model Implementation Steps

In [3]:
### Add the code related to the model implementation steps in cells inside this section
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# Model: a linear SVM, the best classifier in the Master notebook.
# class_weight="balanced" gives the small classes (fear is only 8.6% of the posts)
# more weight, which matters because the metric is MACRO F1.
# C=0.3 (stronger regularization than the default 1.0) was best of 0.3 / 0.5 / 1.0 in CV.
def make_model():
    return LinearSVC(C=0.3, class_weight="balanced", random_state=SEED)

# 1. Judge the model with 5-fold cross-validation on the TRAINING rows only.
#    The public leaderboard scores only part of the test set, and the grade uses the
#    private part, so CV is the honest estimate. The vectorizers are refitted inside
#    every fold, so the held-out fold never leaks into the features.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_pred = np.empty(len(train), dtype=object)       # out-of-fold predictions
fold_scores, tfidf_only_scores = [], []
for fold, (tr, va) in enumerate(cv.split(train["clean"], y), start=1):
    X_tr, X_va = build_features(train["clean"].iloc[tr], train["clean"].iloc[va], E_train[tr], E_train[va])
    oof_pred[va] = make_model().fit(X_tr, y[tr]).predict(X_va)
    fold_scores.append(f1_score(y[va], oof_pred[va], average="macro"))
    # Comparison: the same model on TF-IDF alone, to show what the embeddings add.
    A_tr, A_va = tfidf_blocks(train["clean"].iloc[tr], train["clean"].iloc[va])
    tfidf_only_scores.append(f1_score(y[va], make_model().fit(A_tr, y[tr]).predict(A_va), average="macro"))
    print(f"fold {fold}: macro F1 {fold_scores[-1]:.3f}  (TF-IDF only {tfidf_only_scores[-1]:.3f})")

print(f"\nCV macro F1, TF-IDF + embeddings: {np.mean(fold_scores):.3f} ± {np.std(fold_scores):.3f}")
print(f"CV macro F1, TF-IDF only:         {np.mean(tfidf_only_scores):.3f} ± {np.std(tfidf_only_scores):.3f}")
print("\nPer-emotion scores (out-of-fold predictions):")
print(classification_report(y, oof_pred.astype(str), digits=3))

# 2. Final model: fit on ALL training rows, predict the test rows.
final_model = make_model().fit(X_train, y)
test_pred = pd.Series(final_model.predict(X_test), index=test["post_id"])

# 3. Write the submission in the row order of sampleSubmission.csv.
#    Kaggle's scorer expects the ID column to be called "id" (upload error:
#    "ID column id not found in submission"), even though sampleSubmission.csv
#    names it "post_id". So the column is renamed here.
submission = pd.DataFrame({"id": sample_sub["post_id"],
                           "label": test_pred.loc[sample_sub["post_id"]].to_numpy()})
assert len(submission) == len(sample_sub) and submission["label"].notna().all()
os.makedirs("data/kaggle", exist_ok=True)
submission.to_csv("data/kaggle/submission.csv", index=False)
print("saved data/kaggle/submission.csv:", submission.shape)
print("predicted label share:", submission["label"].value_counts(normalize=True).round(3).to_dict())

fold 1: macro F1 0.566  (TF-IDF only 0.514)


fold 2: macro F1 0.554  (TF-IDF only 0.485)


fold 3: macro F1 0.615  (TF-IDF only 0.519)


fold 4: macro F1 0.563  (TF-IDF only 0.503)


fold 5: macro F1 0.547  (TF-IDF only 0.527)

CV macro F1, TF-IDF + embeddings: 0.569 ± 0.024
CV macro F1, TF-IDF only:         0.510 ± 0.015

Per-emotion scores (out-of-fold predictions):
              precision    recall  f1-score   support

       anger      0.526     0.593     0.557       674
     disgust      0.486     0.390     0.433       454
        fear      0.596     0.562     0.579       288
         joy      0.703     0.735     0.718       875
     sadness      0.590     0.586     0.588       628
    surprise      0.552     0.531     0.541       441

    accuracy                          0.590      3360
   macro avg      0.575     0.566     0.569      3360
weighted avg      0.588     0.590     0.588      3360



saved data/kaggle/submission.csv: (1440, 2)
predicted label share: {'joy': 0.281, 'anger': 0.21, 'sadness': 0.174, 'surprise': 0.128, 'disgust': 0.12, 'fear': 0.087}


### 3.2 Fine-tuned transformers (Kaggle GPU, 5-fold)

The SVM above uses MiniLM and mpnet *as they are*: their weights never change. Fine-tuning is different: it trains
**all** the weights of a pre-trained language model a bit more on our labelled posts, so it learns which phrases show
each emotion. I decided to use only **general** pre-trained models and to avoid any model already trained on emotion
labels, because such a model may have seen these posts.

My laptop has no GPU. The AI wrote the training code below, and I ran it in a **Kaggle notebook with a T4 GPU**
(settings: Accelerator = GPU, Internet = On, competition data added as input). It is shown here, not run here.

* 5-fold cross-validation (seed 42), with each emotion's share the same in every fold. Every training post gets an
  **out-of-fold (OOF)** prediction from a model that never saw it, so all 3,360 posts give an honest score.
  The test prediction is the average of the 5 models.
* AdamW, weight decay 0.01, a learning rate that rises over the first 10% of steps and then falls slowly,
  gradient clipping 1.0, batch 16, max 128 tokens, 4 epochs, mixed precision (fp16) with 32-bit master weights.
* A tokenizer check stops the run if the words come back broken. The AI added it after DeBERTa's first run had a broken tokenizer (see report 1.3).

```python
# Kaggle GPU notebook: 5-fold fine-tuning of a large pre-trained transformer.
# Notebook settings: Accelerator = GPU (T4 x1/x2 or P100), Internet = On,
# and the competition data added as input (the "+ Add Input" button).
# Only GENERAL pre-trained models are used (never one already trained on emotion labels).
import subprocess, sys
# DeBERTa's vocabulary is a SentencePiece file. Without the sentencepiece package,
# transformers silently falls back to a TikToken reader and builds a WRONG tokenizer.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "protobuf"], check=False)
import glob, json, os, random, time
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# (model, learning rate, seed offset). Each entry was run in its own Kaggle GPU session;
# the outputs are saved in data/kaggle/finetune/gpu_train_1 (RoBERTa-large),
# gpu_train_3 (DeBERTa-v3-large) and gpu_train_5 (ELECTRA-base, 3 seeds).
# ELECTRA needs a higher learning rate than RoBERTa.
MODELS = [("roberta-large", 1e-5, 0),
          ("microsoft/deberta-v3-large", 1.5e-5, 0),
          ("google/electra-base-discriminator", 5e-5, 0),
          ("google/electra-base-discriminator", 5e-5, 1),
          ("google/electra-base-discriminator", 5e-5, 2)]
EPOCHS, BATCH, MAX_LEN, SEED, N_FOLDS = 4, 16, 128, 42, 5
OUT = "/kaggle/working"
assert torch.cuda.is_available(), "Turn on the GPU accelerator in the notebook settings."
dev = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

# ---- data: the same merge as the Homework notebook, so the row order matches ----
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found: add the competition data as input"
    return hits[0]
ids = pd.read_csv(find("data_identification.csv"))
labels = pd.read_csv(find("emotion.csv"))
with open(find("final_posts.json"), encoding="utf-8") as f:
    posts = pd.DataFrame(json.load(f))
sample_sub = pd.read_csv(find("sampleSubmission.csv"))
df = ids.merge(posts, on="post_id", how="left").merge(labels, on="post_id", how="left")
train = df[df.set == "train"].reset_index(drop=True)
test = df[df.set == "test"].reset_index(drop=True)
CLASSES = sorted(train.label.unique())
y = train.label.map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
texts, test_texts = train.text.tolist(), test.text.tolist()
folds = np.zeros(len(train), dtype=int)
for k, (_, va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(texts, y)):
    folds[va] = k
np.save(f"{OUT}/fold_id.npy", folds)
json.dump(CLASSES, open(f"{OUT}/classes.json", "w"))
print("train", len(train), "| test", len(test), "| classes", CLASSES)

def run_model(model_name, lr, offset=0):
    tag = f'{model_name.split("/")[-1]}-s{offset}'
    tok = AutoTokenizer.from_pretrained(model_name)
    # Sanity check: a correct tokenizer gives readable word pieces and decodes back to the text.
    sample = "I can't believe it, that is disgusting!"
    print("tokenizer:", type(tok).__name__, "| vocab", len(tok), "|", tok.tokenize(sample))
    back = tok.decode(tok(sample)["input_ids"], skip_special_tokens=True)
    assert back.replace(" ", "").lower() == sample.replace(" ", "").lower(), f"BROKEN TOKENIZER: {back!r}"
    ids_ = tok(texts[:500])["input_ids"]
    unk = sum(i == tok.unk_token_id for x in ids_ for i in x) / sum(len(x) for x in ids_)
    assert unk < 0.01, f"BROKEN TOKENIZER: {unk:.1%} unknown tokens"
    print(f"tokenizer OK (unknown-token rate {unk:.2%})")
    lengths = np.array([len(x) for x in tok(texts, truncation=True, max_length=MAX_LEN)["input_ids"]])
    encode = lambda t: tok(t, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev)

    def batches(idx, rng):
        idx = idx[rng.permutation(len(idx))]                      # shuffle
        out = []
        for g in [idx[i:i + BATCH * 50] for i in range(0, len(idx), BATCH * 50)]:
            g = g[np.argsort(lengths[g], kind="stable")]           # similar lengths together = less padding
            out += [g[i:i + BATCH] for i in range(0, len(g), BATCH)]
        rng.shuffle(out)
        return out

    @torch.no_grad()
    def predict(model, tlist):
        model.eval()
        probs = np.zeros((len(tlist), len(CLASSES)), dtype=np.float32)
        order = np.argsort([len(t) for t in tlist])
        for i in range(0, len(order), 64):
            sel = order[i:i + 64]
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**encode([tlist[j] for j in sel])).logits
            probs[sel] = torch.softmax(logits.float(), -1).cpu().numpy()
        return probs

    oof = np.zeros((EPOCHS, len(train), len(CLASSES)), dtype=np.float32)   # out-of-fold, every epoch
    tst = np.zeros((EPOCHS, len(test), len(CLASSES)), dtype=np.float32)    # test, averaged over folds
    for k in range(N_FOLDS):
        tr, va = np.where(folds != k)[0], np.where(folds == k)[0]
        for attempt in range(2):                                   # retry once if training collapses
            seed = SEED + k + 100 * attempt + 1000 * offset
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
            rng = np.random.default_rng(seed)
            model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CLASSES))
            model = model.float().to(dev)      # weights must be 32-bit for GradScaler (DeBERTa loads as 16-bit)
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
            n_steps = EPOCHS * int(np.ceil(len(tr) / BATCH))
            sched = get_linear_schedule_with_warmup(opt, int(0.1 * n_steps), n_steps)
            scaler = torch.amp.GradScaler("cuda")
            fold_oof, fold_tst, t0 = [], [], time.time()
            for ep in range(EPOCHS):
                model.train()
                for b in batches(tr, rng):
                    with torch.autocast("cuda", dtype=torch.float16):
                        logits = model(**encode([texts[j] for j in b])).logits
                    loss = F.cross_entropy(logits.float(), torch.tensor(y[b], device=dev))
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
                fold_oof.append(predict(model, [texts[j] for j in va]))
                fold_tst.append(predict(model, test_texts))
                f1 = f1_score(y[va], fold_oof[-1].argmax(1), average="macro")
                print(f"{tag} fold {k} epoch {ep + 1}: val macro F1 {f1:.4f} ({time.time() - t0:.0f}s)", flush=True)
            if f1 > 0.30:
                break
            print(f"{tag} fold {k}: collapsed (F1 {f1:.3f}), retrying with another seed", flush=True)
        for ep in range(EPOCHS):
            oof[ep, va] = fold_oof[ep]
            tst[ep] += fold_tst[ep] / N_FOLDS
        del model; torch.cuda.empty_cache()

    np.save(f"{OUT}/oof_{tag}.npy", oof)
    np.save(f"{OUT}/test_{tag}.npy", tst)
    scores = [f1_score(y, oof[ep].argmax(1), average="macro") for ep in range(EPOCHS)]
    print(f"\n{tag} OUT-OF-FOLD macro F1 per epoch (all {len(train)} posts): {np.round(scores, 4).tolist()}")
    ep = int(np.argmax(scores))
    pred = pd.Series(np.array(CLASSES)[tst[ep].argmax(1)], index=test.post_id)
    sub = pd.DataFrame({"id": sample_sub.post_id, "label": pred.loc[sample_sub.post_id].to_numpy()})
    sub.to_csv(f"{OUT}/submission_{tag}.csv", index=False)
    print(f"saved submission_{tag}.csv (epoch {ep + 1})")

for model_name, lr, offset in MODELS:
    try:
        run_model(model_name, lr, offset)
    except Exception as e:                                         # e.g. a tokenizer problem
        print(f"{model_name} failed: {e!r}")
print("ALL DONE. Download everything in /kaggle/working (Output tab).")
```


In [4]:
# Load the Kaggle GPU outputs: out-of-fold (OOF) and test probabilities for every epoch.
# Shapes: OOF (epochs, 3360, 6), test (epochs, 1440, 6), rows in the same order as `train` / `test`.
import glob
from sklearn.metrics import f1_score

CLASSES = sorted(train["label"].unique())                     # same alphabetical order as on Kaggle
y_int = train["label"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
FT = "data/kaggle/finetune"

folds = np.load(f"{FT}/gpu_train_1/fold_id.npy")             # fold of each training post
for d in ["gpu_train_3", "gpu_train_5"]:
    assert (np.load(f"{FT}/{d}/fold_id.npy") == folds).all()  # every run used the same 5 folds

def load_runs(folder):
    """Last-epoch probabilities, averaged over all seeds saved in `folder`."""
    files = sorted(glob.glob(f"{FT}/{folder}/oof_*.npy"))
    oof = np.mean([np.load(f)[-1] for f in files], axis=0)
    test_p = np.mean([np.load(f.replace("oof_", "test_"))[-1] for f in files], axis=0)
    return oof, test_p

models = {"RoBERTa-large": load_runs("gpu_train_1"),
          "ELECTRA-base x3": load_runs("gpu_train_5"),
          "DeBERTa-v3-large": load_runs("gpu_train_3")}
macro = lambda probs, rows=slice(None): f1_score(y_int[rows], probs[rows].argmax(1), average="macro")
for name, (oof, _) in models.items():
    print(f"{name:17s} OOF macro F1 on all {len(train)} posts: {macro(oof):.4f}")

RoBERTa-large     OOF macro F1 on all 3360 posts: 0.6023
ELECTRA-base x3   OOF macro F1 on all 3360 posts: 0.6001
DeBERTa-v3-large  OOF macro F1 on all 3360 posts: 0.5583


### 3.3 Final blend and submission

The SVM and the fine-tuned models make some different mistakes, so a weighted average of their probabilities is
better than each one alone. The SVM gets out-of-fold scores on the **same 5 folds**, so every model is judged on all
3,360 posts. SVM scores are not probabilities, so they are turned into probability-like numbers with
`softmax(2 * score)`. The code below (written and run by the AI) picks the weights from a small grid by OOF macro F1.
Then a fair check (pick the weights on 3 folds, score them on the other 2) measures how much that choice makes the
score look too good.

In [5]:
import itertools

def softmax(z, T):
    z = z * T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

# SVM out-of-fold scores on the same folds (vectorizers refitted inside every fold, no leakage).
svm_oof = np.zeros((len(train), len(CLASSES)))
for k in range(5):
    tr, va = np.where(folds != k)[0], np.where(folds == k)[0]
    X_tr, X_va = build_features(train["clean"].iloc[tr], train["clean"].iloc[va], E_train[tr], E_train[va])
    svm_oof[va] = make_model().fit(X_tr, y_int[tr]).decision_function(X_va)
svm_test = make_model().fit(X_train, y_int).decision_function(X_test)
models["SVM"] = (softmax(svm_oof, 2), softmax(svm_test, 2))
print(f"{'SVM':17s} OOF macro F1 on all {len(train)} posts: {macro(models['SVM'][0]):.4f}")

# Small grid of relative weights (normalised to sum to 1).
names = list(models)
grid = {"RoBERTa-large": [0, 1, 2], "ELECTRA-base x3": [0, 1, 2], "DeBERTa-v3-large": [0, 0.5, 1], "SVM": [0, 0.5, 1]}
combos = [np.array(w, float) / sum(w) for w in itertools.product(*[grid[n] for n in names]) if sum(w) > 0]
blend = lambda w, part: sum(wi * models[n][part] for wi, n in zip(w, names) if wi > 0)   # part 0 = OOF, 1 = test

scores = np.array([macro(blend(w, 0)) for w in combos])
order = np.argsort(-scores)
print(f"\n{len(combos)} weight combinations. Top 3 by OOF macro F1:")
for i in order[:3]:
    print(f"  {scores[i]:.4f}  " + ", ".join(f"{n} {w:.2f}" for n, w in zip(names, combos[i]) if w > 0))

# Nested check: choose the best combination on 3 folds, score it on the other 2.
gaps = []
for held in itertools.combinations(range(5), 2):
    B = np.isin(folds, held)
    on_A = [macro(blend(w, 0), ~B) for w in combos]
    best_i = int(np.argmax(on_A))
    gaps.append(on_A[best_i] - macro(blend(combos[best_i], 0), B))
best_w = combos[order[0]]
print(f"optimism of picking the best: {np.mean(gaps):+.4f} -> honest estimate ~{scores[order[0]] - max(np.mean(gaps), 0):.3f}")

print("\nPer-emotion scores of the final blend (out-of-fold):")
print(classification_report(y_int, blend(best_w, 0).argmax(1), target_names=CLASSES, digits=3))

# Final submission in sampleSubmission order, with the "id" column Kaggle expects.
final_pred = pd.Series(np.array(CLASSES)[blend(best_w, 1).argmax(1)], index=test["post_id"])
final_sub = pd.DataFrame({"id": sample_sub["post_id"], "label": final_pred.loc[sample_sub["post_id"]].to_numpy()})
assert len(final_sub) == len(sample_sub) and final_sub["label"].notna().all()
final_sub.to_csv("data/kaggle/submission_final_v2.csv", index=False)
print("saved data/kaggle/submission_final_v2.csv | weights:",
      {n: round(float(w), 2) for n, w in zip(names, best_w) if w > 0})

SVM               OOF macro F1 on all 3360 posts: 0.5694

80 weight combinations. Top 3 by OOF macro F1:
  0.6190  RoBERTa-large 0.29, ELECTRA-base x3 0.29, DeBERTa-v3-large 0.14, SVM 0.29
  0.6187  RoBERTa-large 0.25, ELECTRA-base x3 0.50, SVM 0.25
  0.6181  RoBERTa-large 0.33, ELECTRA-base x3 0.33, SVM 0.33


optimism of picking the best: +0.0052 -> honest estimate ~0.614

Per-emotion scores of the final blend (out-of-fold):
              precision    recall  f1-score   support

       anger      0.577     0.662     0.616       674
     disgust      0.526     0.374     0.438       454
        fear      0.713     0.587     0.644       288
         joy      0.750     0.790     0.769       875
     sadness      0.615     0.678     0.645       628
    surprise      0.622     0.583     0.602       441

    accuracy                          0.643      3360
   macro avg      0.634     0.612     0.619      3360
weighted avg      0.640     0.643     0.638      3360

saved data/kaggle/submission_final_v2.csv | weights: {'RoBERTa-large': 0.29, 'ELECTRA-base x3': 0.29, 'DeBERTa-v3-large': 0.14, 'SVM': 0.29}
